In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, TimestampType

In [3]:
spark = (SparkSession.builder.appName("cloudBilling").getOrCreate())

In [4]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("resource_id", StringType(), False),
    StructField("service_name", StringType(), True),
    StructField("usage_quantity", DoubleType(), True),
    StructField("usage_unit", StringType(), True),
    StructField("region_zone", StringType(), True),
    StructField("cpu_utilization", DoubleType(), True),
    StructField("memory_utilization", DoubleType(), True),
    StructField("network_inbound", DoubleType(), True),
    StructField("network_outbound", DoubleType(), True),
    StructField("usage_start_date", StringType(), True),
    StructField("usage_end_date", StringType(), True),
    StructField("cost_per_quantity", DoubleType(), True),
    StructField("unrounded_cost", DoubleType(), True),
    StructField("rounded_cost", DoubleType(), True),
    StructField("total_cost", DoubleType(), True)
])


In [5]:
df = spark.read.option("header",True).option("inferSchema", True).schema(schema).csv("hdfs://hdfs-namenode:9000/data/raw/sales/ingest_date=2026-01-21")

In [6]:
df.printSchema()

root
 |-- resource_id: string (nullable = true)
 |-- service_name: string (nullable = true)
 |-- usage_quantity: double (nullable = true)
 |-- usage_unit: string (nullable = true)
 |-- region_zone: string (nullable = true)
 |-- cpu_utilization: double (nullable = true)
 |-- memory_utilization: double (nullable = true)
 |-- network_inbound: double (nullable = true)
 |-- network_outbound: double (nullable = true)
 |-- usage_start_date: string (nullable = true)
 |-- usage_end_date: string (nullable = true)
 |-- cost_per_quantity: double (nullable = true)
 |-- unrounded_cost: double (nullable = true)
 |-- rounded_cost: double (nullable = true)
 |-- total_cost: double (nullable = true)



In [7]:
print(f"Total rows: {df.count()}")

Total rows: 1000


In [8]:
df = df.withColumn("start_ts", F.to_timestamp(F.col("usage_start_date"),"MM-dd-yyyy HH:mm")) \
        .withColumn("end_ts", F.to_timestamp(F.col("usage_end_date"),"MM-dd-yyyy HH:mm"))


In [55]:
df.show(5, truncate=False)

+------------+---------------+--------------+----------+------------------+---------------+------------------+---------------+----------------+----------------+----------------+-----------------+--------------+------------+----------+-------------------+-------------------+
|resource_id |service_name   |usage_quantity|usage_unit|region_zone       |cpu_utilization|memory_utilization|network_inbound|network_outbound|usage_start_date|usage_end_date  |cost_per_quantity|unrounded_cost|rounded_cost|total_cost|start_ts           |end_ts             |
+------------+---------------+--------------+----------+------------------+---------------+------------------+---------------+----------------+----------------+----------------+-----------------+--------------+------------+----------+-------------------+-------------------+
|res-ST6BAJ2N|Cloud Dataproc |954.9843      |Requests  |europe-north1     |92.72          |70.42             |7.7097035547E10|8.2685933273E10 |01-08-2024 22:24|07-08-2024 06:5

In [56]:
df = df.withColumn("usage_duration_hours",(F.col("end_ts").cast("long") - F.col("start_ts").cast("long"))/3600)

In [57]:
df.show(5,truncate=False)

+------------+---------------+--------------+----------+------------------+---------------+------------------+---------------+----------------+----------------+----------------+-----------------+--------------+------------+----------+-------------------+-------------------+--------------------+
|resource_id |service_name   |usage_quantity|usage_unit|region_zone       |cpu_utilization|memory_utilization|network_inbound|network_outbound|usage_start_date|usage_end_date  |cost_per_quantity|unrounded_cost|rounded_cost|total_cost|start_ts           |end_ts             |usage_duration_hours|
+------------+---------------+--------------+----------+------------------+---------------+------------------+---------------+----------------+----------------+----------------+-----------------+--------------+------------+----------+-------------------+-------------------+--------------------+
|res-ST6BAJ2N|Cloud Dataproc |954.9843      |Requests  |europe-north1     |92.72          |70.42             |7.

In [58]:
numeric_cols = ["cpu_utilization", "memory_utilization","network_inbound", "network_outbound", "usage_quantity"]

In [59]:
df = df.fillna(0, subset=numeric_cols)

In [61]:
from pyspark.sql.functions import *

df = df.withColumn("cost_per_hour", 
                          col("total_cost") / col("usage_duration_hours")) \
               .withColumn("network_total", 
                          col("network_inbound") + col("network_outbound")) \
               .withColumn("resource_efficiency", 
                          (col("cpu_utilization") + col("memory_utilization")) / 2) \
               .withColumn("cost_efficiency", 
                          col("usage_quantity") / (col("total_cost") + 0.001))

In [62]:
df.show(5, truncate=False)

+------------+---------------+--------------+----------+------------------+---------------+------------------+---------------+----------------+----------------+----------------+-----------------+--------------+------------+----------+-------------------+-------------------+--------------------+-----------------+----------------+-------------------+---------------------+
|resource_id |service_name   |usage_quantity|usage_unit|region_zone       |cpu_utilization|memory_utilization|network_inbound|network_outbound|usage_start_date|usage_end_date  |cost_per_quantity|unrounded_cost|rounded_cost|total_cost|start_ts           |end_ts             |usage_duration_hours|cost_per_hour    |network_total   |resource_efficiency|cost_efficiency      |
+------------+---------------+--------------+----------+------------------+---------------+------------------+---------------+----------------+----------------+----------------+-----------------+--------------+------------+----------+-------------------+

In [63]:
 df = df.withColumn("hour_of_day", col("start_ts").cast("int") % 86400 / 3600) \
        .withColumn("day_of_week", (datediff(col("start_ts"), lit("1970-01-01")) % 7))


In [64]:
output_path = "hdfs://hdfs-namenode:9000/data/feature_engineering"

In [66]:
df.write.mode("overwrite").partitionBy("usage_unit").parquet(output_path)

In [70]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans

In [71]:
feature_cols = ["cpu_utilization", "memory_utilization", "network_total",
                       "cost_per_hour", "usage_quantity", "resource_efficiency"]

In [72]:
assembler = VectorAssembler(inputCols=feature_cols,outputCol="features_raw",handleInvalid="skip")

In [73]:
df.show(5,truncate=False)

+------------+---------------+--------------+----------+------------------+---------------+------------------+---------------+----------------+----------------+----------------+-----------------+--------------+------------+----------+-------------------+-------------------+--------------------+-----------------+----------------+-------------------+---------------------+-----------+-----------+
|resource_id |service_name   |usage_quantity|usage_unit|region_zone       |cpu_utilization|memory_utilization|network_inbound|network_outbound|usage_start_date|usage_end_date  |cost_per_quantity|unrounded_cost|rounded_cost|total_cost|start_ts           |end_ts             |usage_duration_hours|cost_per_hour    |network_total   |resource_efficiency|cost_efficiency      |hour_of_day|day_of_week|
+------------+---------------+--------------+----------+------------------+---------------+------------------+---------------+----------------+----------------+----------------+-----------------+-----------

In [74]:
df_assembled = assembler.transform(df)

In [76]:
scaler = StandardScaler(inputCol="features_raw", 
                               outputCol="features",
                               withStd=True, 
                               withMean=True)

In [77]:
scaler_model = scaler.fit(df_assembled)

In [78]:
df_scaled = scaler_model.transform(df_assembled)

In [80]:
kmeans = KMeans(k=10, seed=42, maxIter=20, 
                       featuresCol="features", 
                       predictionCol="cluster")

In [81]:
model = kmeans.fit(df_scaled)

In [82]:
df_clustered = model.transform(df_scaled)

In [93]:
# from pyspark.sql.functions import lit
# df_clustered = df_clustered.withColumn("cluster_distance", 
#                                                lit(model.summary.trainingCost / 
#                                                df_clustered.count()))

In [94]:
# distance_threshold = df_clustered.approxQuantile("cluster_distance", [0.95], 0.01)[0]

In [95]:
centers = model.clusterCenters()

In [96]:
import numpy as np
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

def distance_to_center(features, cluster):
    center = centers[cluster]
    return float(np.linalg.norm(features - center))

distance_udf = udf(distance_to_center, DoubleType())

df_clustered = df_clustered.withColumn(
    "cluster_distance",
    distance_udf(col("features"), col("cluster"))
)


In [97]:
threshold = df_clustered.approxQuantile(
    "cluster_distance", [0.95], 0.01
)[0]

In [98]:
df_clustered = df_clustered.withColumn(
    "ml_anomaly",
    when(col("cluster_distance") > threshold, 1)
    .otherwise(0)
)

In [89]:
df_anamoly = df_clustered.filter(col("ml_anomaly") == 1)

In [99]:
df_clustered.groupBy("ml_anomaly").count().show()

+----------+-----+
|ml_anomaly|count|
+----------+-----+
|         1|   17|
|         0|  276|
+----------+-----+



In [101]:
output_path = "hdfs://hdfs-namenode:9000/data/feature_engineering/anomaly"

In [102]:
df_clustered.write.mode("overwrite").partitionBy("usage_unit").parquet(output_path)

In [103]:
anomaly_row = df_clustered.filter(F.col("ml_anomaly") == 1)

In [1]:
# Display the key metrics for this specific anomaly
anomaly_row.select(
  "resource_id","service_name", "cpu_utilization", "memory_utilization", "network_total","cost_per_hour", "usage_quantity", "resource_efficiency"
).show()

NameError: name 'anomaly_row' is not defined

In [11]:
df = spark.read.parquet(
    "hdfs://hdfs-namenode:9000/data/feature_engineering/anomaly"
)

In [12]:
df.show(5)

+------------+------------+--------------+------------------+---------------+------------------+---------------+----------------+----------------+----------------+-----------------+--------------+------------+----------+-------------------+-------------------+--------------------+------------------+----------------+-------------------+--------------------+------------------+-----------+--------------------+--------------------+-------+------------------+----------+----------+
| resource_id|service_name|usage_quantity|       region_zone|cpu_utilization|memory_utilization|network_inbound|network_outbound|usage_start_date|  usage_end_date|cost_per_quantity|unrounded_cost|rounded_cost|total_cost|           start_ts|             end_ts|usage_duration_hours|     cost_per_hour|   network_total|resource_efficiency|     cost_efficiency|       hour_of_day|day_of_week|        features_raw|            features|cluster|  cluster_distance|ml_anomaly|usage_unit|
+------------+------------+-----------

In [15]:
anomaly_row = df.filter(F.col("ml_anomaly") == 1)

anomaly_row.select(
  "resource_id","service_name", "cpu_utilization", "memory_utilization", "network_total","cost_per_hour", "usage_quantity", "resource_efficiency"
).show()

+------------+------------------+---------------+------------------+----------------+------------------+--------------+-------------------+
| resource_id|      service_name|cpu_utilization|memory_utilization|   network_total|     cost_per_hour|usage_quantity|resource_efficiency|
+------------+------------------+---------------+------------------+----------------+------------------+--------------+-------------------+
|res-HLWAGRJX|           Pub/Sub|           5.46|             98.89|1.79613244554E11| 471.9478269014612|      565.1884|             52.175|
|res-J1A4NFH8|         Cloud VPN|          32.32|             14.03|2.00278223452E11|277.14723225305113|      789.3788|             23.175|
|res-SJ11891I|         Cloud CDN|          11.67|              6.75| 8.2666822335E10| 305.4041714226419|      920.1814|               9.21|
|res-GL3BFL4Z|Cloud Interconnect|          99.65|              0.11|   4.484673259E9|1.7325800376647833|       30.4465|              49.88|
|res-13MDNGII|      